In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Load the Student Attitude and Behavior dataset
df = pd.read_csv(r'c:\vscode\CVprojects\kenexaiHackathon\datasets\Student Attitude and Behavior.csv')
print("Original Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

Original Dataset Shape: (235, 19)

First few rows:
  Certification Course  Gender Department  Height(CM)  Weight(KG)  10th Mark  \
0                   No    Male        BCA       100.0        58.0       79.0   
1                   No  Female        BCA        90.0        40.0       70.0   
2                  Yes    Male        BCA       159.0        78.0       71.0   
3                  Yes  Female        BCA       147.0        20.0       70.0   
4                   No    Male        BCA       170.0        54.0       40.0   

   12th Mark  college mark        hobbies daily studing time  \
0       64.0          80.0    Video Games      0 - 30 minute   
1       80.0          70.0         Cinema     30 - 60 minute   
2       61.0          55.0         Cinema         1 - 2 Hour   
3       59.0          58.0  Reading books         1 - 2 Hour   
4       65.0          30.0    Video Games     30 - 60 minute   

  prefer to study in  salary expectation Do you like your degree?  \
0            M

In [17]:
# STEP 1: Clean column names
# Remove extra spaces, convert to lowercase, replace spaces with underscore
df.columns = df.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
print("Cleaned columns:")
print(df.columns.tolist())

Cleaned columns:
['certification_course', 'gender', 'department', 'height_cm', 'weight_kg', '10th_mark', '12th_mark', 'college_mark', 'hobbies', 'daily_studing_time', 'prefer_to_study_in', 'salary_expectation', 'do_you_like_your_degree', 'willingness_to_pursue_a_career_based_on_their_degree', 'social_medai_video', 'travelling_time', 'stress_level', 'financial_status', 'part_time_job']


In [2]:
# STEP 2: Check missing values and duplicates
print("="*60)
print("DATASET INFO")
print("="*60)
print(f"Shape: {df.shape}")
print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")

DATASET INFO
Shape: (235, 19)

Duplicates: 0

Missing values:
Certification Course                                      0
Gender                                                    0
Department                                                0
Height(CM)                                                0
Weight(KG)                                                0
10th Mark                                                 0
12th Mark                                                 0
college mark                                              0
hobbies                                                   0
daily studing time                                        0
prefer to study in                                        0
salary expectation                                        0
Do you like your degree?                                  0
willingness to pursue a career based on their degree      0
social medai & video                                      0
Travelling Time                       

In [3]:
# Check actual column names
print("All columns in dataset:")
print(df.columns.tolist())
print(f"\nTotal columns: {len(df.columns)}")


All columns in dataset:
['Certification Course', 'Gender', 'Department', 'Height(CM)', 'Weight(KG)', '10th Mark', '12th Mark', 'college mark', 'hobbies', 'daily studing time', 'prefer to study in', 'salary expectation', 'Do you like your degree?', 'willingness to pursue a career based on their degree  ', 'social medai & video', 'Travelling Time ', 'Stress Level ', 'Financial Status', 'part-time job']

Total columns: 19


In [4]:
# STEP 3: Convert numeric columns properly and handle missing values
numeric_cols = ['height_cm', 'weight_kg', '10th_mark', '12th_mark', 'college_mark']
categorical_cols = ['gender', 'department', 'hobbies', 'prefer_to_study_in', 'financial_status']
ordinal_cols = ['daily_studying_time', 'stress_level', 'salary_expectation']
binary_cols = ['certification_course', 'do_you_like_your_degree', 
               'willingness_to_pursue_career_based_on_degree', 'part_time_job']

# Remove duplicates first
df = df.drop_duplicates().reset_index(drop=True)

# Filter and convert numeric columns that exist
numeric_cols_existing = [col for col in numeric_cols if col in df.columns]
for col in numeric_cols_existing:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print("Numeric columns converted and missing values filled with median")
print(f"Numeric columns processed: {numeric_cols_existing}")
print(f"Shape after handling: {df.shape}")

Numeric columns converted and missing values filled with median
Numeric columns processed: []
Shape after handling: (235, 19)


In [5]:
# STEP 4: Clean categorical columns
# Filter categorical_cols to only include columns that exist
categorical_cols_existing = [col for col in categorical_cols if col in df.columns]

# Strip whitespace and fill missing with mode for categorical columns
for col in categorical_cols_existing:
    # Strip whitespace from string values
    df[col] = df[col].astype(str).str.strip()
    # Fill missing values with mode
    if df[col].isnull().sum() > 0 or (df[col] == 'nan').sum() > 0:
        mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
        df[col] = df[col].replace(['nan', 'None', '', 'NaN'], mode_val)
        df[col].fillna(mode_val, inplace=True)

# Convert categorical columns to category dtype
for col in categorical_cols_existing:
    df[col] = df[col].astype('category')

print("Categorical columns cleaned and converted to category dtype")
if categorical_cols_existing:
    print(f"Columns processed: {categorical_cols_existing}")
    print(f"Category value counts:\n{df[categorical_cols_existing].nunique()}")
else:
    print("No categorical columns found in dataset")

Categorical columns cleaned and converted to category dtype
Columns processed: ['hobbies']
Category value counts:
hobbies    4
dtype: int64


In [6]:
# STEP 5: Convert ordinal columns into numeric scores
# Define mappings for ordinal columns - only process columns that exist
ordinal_mappings = {
    'daily_studying_time': {
        '0-1': 1, '1-2': 2, '2-3': 3, '3-4': 4, '4-5': 5, '5+': 6,
        '0': 1, '1': 2, '2': 3, '3': 4, '4': 5, '5': 6
    },
    'stress_level': {
        'low': 1, 'medium': 2, 'high': 3, 'very high': 4,
        'very low': 1, 'moderate': 2
    },
    'salary_expectation': {
        '0-2': 1, '2-4': 2, '4-6': 3, '6-8': 4, '8-10': 5, '10+': 6,
        '0': 1, '2': 2, '4': 3, '6': 4, '8': 5, '10': 6
    },
    'travelling_time': {
        '0-30': 1, '30-60': 2, '60-120': 3, '120+': 4,
        '0-30 minutes': 1, '30-60 minutes': 2, '60-120 minutes': 3, '120+ minutes': 4,
        '0': 1, '30': 2, '60': 3, '120': 4
    }
}

# Filter ordinal_cols to only include columns that exist
ordinal_cols_existing = [col for col in ordinal_cols if col in df.columns]

# Also check for travelling_time even though not in original list
if 'travelling_time' in df.columns and 'travelling_time' not in ordinal_cols_existing:
    ordinal_cols_existing.append('travelling_time')

for col in ordinal_cols_existing:
    df[col] = df[col].astype(str).str.strip().str.lower()
    if col in ordinal_mappings:
        df[col] = df[col].map(ordinal_mappings[col])
    # Ensure numeric type
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Fill any missing with median
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print("Ordinal columns converted to numeric scores")
if ordinal_cols_existing:
    print(f"Ordinal columns processed: {ordinal_cols_existing}")
    print(f"Summary:\n{df[ordinal_cols_existing].describe()}")
else:
    print("No ordinal columns found in dataset")

Ordinal columns converted to numeric scores
No ordinal columns found in dataset


In [7]:
# STEP 6: Convert Yes/No columns to binary (1/0)
yes_no_mapping = {'yes': 1, 'no': 0, 'y': 1, 'n': 0, '1': 1, '0': 0, 'true': 1, 'false': 0}

# Filter binary_cols to only include columns that exist
binary_cols_existing = [col for col in binary_cols if col in df.columns]

for col in binary_cols_existing:
    # Convert to lowercase and map
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].map(yes_no_mapping)
    # Ensure numeric type
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Fill missing with mode (0 or 1)
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 0
        df[col].fillna(mode_val, inplace=True)

print("Binary columns converted to 1/0")
if binary_cols_existing:
    print(f"Binary columns processed: {binary_cols_existing}")
    for col in binary_cols_existing:
        print(f"{col}: {df[col].value_counts().to_dict()}")
else:
    print("No binary columns found in dataset")

Binary columns converted to 1/0
No binary columns found in dataset


In [24]:
# STEP 7: Feature Engineering
# Create BMI = weight_kg / (height_cm/100)^2
if 'height_cm' in df.columns and 'weight_kg' in df.columns:
    df['bmi'] = df['weight_kg'] / ((df['height_cm'] / 100) ** 2)
    print(f"BMI created - Mean: {df['bmi'].mean():.2f}, Std: {df['bmi'].std():.2f}")

# Create academic_average = (10th_mark + 12th_mark + college_mark)/3
if all(col in df.columns for col in ['10th_mark', '12th_mark', 'college_mark']):
    df['academic_average'] = (df['10th_mark'] + df['12th_mark'] + df['college_mark']) / 3
    print(f"Academic Average created - Mean: {df['academic_average'].mean():.2f}, Std: {df['academic_average'].std():.2f}")

print("\nNew features created!")
print(f"Updated shape: {df.shape}")

BMI created - Mean: 113.80, Std: 1351.36
Academic Average created - Mean: 72.09, Std: 10.61

New features created!
Updated shape: (235, 21)


In [8]:
# STEP 8: Normalize numeric columns using MinMaxScaler (0-1 range)
scaler = MinMaxScaler()

# List of columns to normalize
normalize_cols = ['height_cm', 'weight_kg', 'academic_average', 'travelling_time', 'bmi']
normalize_cols = [col for col in normalize_cols if col in df.columns]

if normalize_cols:
    df[normalize_cols] = scaler.fit_transform(df[normalize_cols])
    print(f"Normalized columns: {normalize_cols}")
    print(f"\nNormalized data range (should be 0-1):")
    print(df[normalize_cols].describe())
else:
    print("No normalization columns found")

No normalization columns found


In [26]:
# STEP 9: Remove irrelevant columns
# Remove hobbies if present (optional - can be kept for analysis)
cols_to_drop = []
if 'hobbies' in df.columns:
    cols_to_drop.append('hobbies')

if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped columns: {cols_to_drop}")
else:
    print("No columns to drop")

print(f"\nFinal shape: {df.shape}")
print(f"Final columns: {df.columns.tolist()}")

Dropped columns: ['hobbies']

Final shape: (235, 20)
Final columns: ['certification_course', 'gender', 'department', 'height_cm', 'weight_kg', '10th_mark', '12th_mark', 'college_mark', 'daily_studing_time', 'prefer_to_study_in', 'salary_expectation', 'do_you_like_your_degree', 'willingness_to_pursue_a_career_based_on_their_degree', 'social_medai_video', 'travelling_time', 'stress_level', 'financial_status', 'part_time_job', 'bmi', 'academic_average']


In [9]:
# STEP 10: Save cleaned dataset
output_dir = Path(r'c:\vscode\CVprojects\kenexaiHackathon\cleaned datasets')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'cleaned_attitude_dataset.csv'
df.to_csv(output_path, index=False)

print("="*60)
print("PREPROCESSING COMPLETE!")
print("="*60)
print(f"✓ Cleaned dataset saved to: {output_path}")
print(f"✓ Final shape: {df.shape}")
print(f"✓ Columns: {df.columns.tolist()}")
print(f"\nDataset Info:")
print(df.info())

PREPROCESSING COMPLETE!
✓ Cleaned dataset saved to: c:\vscode\CVprojects\kenexaiHackathon\cleaned datasets\cleaned_attitude_dataset.csv
✓ Final shape: (235, 19)
✓ Columns: ['Certification Course', 'Gender', 'Department', 'Height(CM)', 'Weight(KG)', '10th Mark', '12th Mark', 'college mark', 'hobbies', 'daily studing time', 'prefer to study in', 'salary expectation', 'Do you like your degree?', 'willingness to pursue a career based on their degree  ', 'social medai & video', 'Travelling Time ', 'Stress Level ', 'Financial Status', 'part-time job']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 19 columns):
 #   Column                                                  Non-Null Count  Dtype   
---  ------                                                  --------------  -----   
 0   Certification Course                                    235 non-null    object  
 1   Gender                                                  235 non-n